# Welcome to Colab!

# Credit Risk Assessment & Loan Optimization System

Transforming 8-Month Customer Credit Data into a Business Decision Support Dashboard

## Project Overview

This project focuses on analyzing customer credit behavior over 8 months
to build a customer-level risk assessment and loan optimization system.

The objective is to transform raw transactional data into actionable
insights that can help financial institutions:

- Identify high-risk customers
- Understand risk-driving factors
- Optimize loan approval strategies
- Reduce default probability

The final output of this project is a Power BI dashboard
built using a cleaned and engineered dataset prepared in Python.

## Problem Statement

Financial institutions face significant risk while approving loans.
Incorrect risk evaluation may result in high defaults and financial losses.

Given 8 months of customer-level financial and credit behavior data,
the goal is to:

- Analyze customer risk patterns
- Identify key factors influencing credit score
- Build a risk segmentation framework
- Provide decision-support insights for loan optimization

## Dataset Description

The dataset contains 100,000 records of customers over 8 months.

Each customer has monthly records including:

- Income details
- Credit utilization
- Outstanding debt
- EMI payments
- Delayed payment history
- Credit inquiries
- Loan details
- Credit score category (Good, Standard, Poor)

Since each customer appears 8 times (one per month),
aggregation was performed to create a customer-level risk profile.

## Key Attributes

The dataset includes the following important attributes:

- Annual_Income → Customer’s yearly income
- Monthly_Inhand_Salary → Net monthly salary
- Credit_Utilization_Ratio → Percentage of credit used
- Outstanding_Debt → Total debt amount
- Total_EMI_per_month → Monthly loan repayment amount
- Delay_from_due_date → Average payment delay
- Num_of_Delayed_Payment → Total delayed payments
- Num_Credit_Inquiries → Number of credit inquiries
- Credit_Score → Credit category (Good / Standard / Poor)

These attributes are critical in evaluating customer credit risk.

## Data Cleaning Process

The following preprocessing steps were performed:

- Removed unnecessary columns (Name, SSN, ID)
- Handled missing values
- Removed duplicates
- Standardized categorical values (e.g., Credit Score)

This ensured data consistency and reliability before analysis.

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
df= pd.read_csv("/content/dataset (1).csv")

In [ ]:
# Remove unnecessary columns

df = df.drop(columns=['Name', 'SSN', 'ID'], errors='ignore')



In [ ]:
# count of duplicate values
df.duplicated().sum()


np.int64(0)

In [ ]:
# Missing Values/Null Values Count
df.isnull().sum()

,0
Customer_ID,0
Month,0
Age,0
Occupation,0
Annual_Income,0
Monthly_Inhand_Salary,0
Num_Bank_Accounts,0
Num_Credit_Card,0
Interest_Rate,0
Num_of_Loan,0


In [ ]:
df = df.dropna()

Since only 1 record contained missing values (<0.001% of dataset),
it was removed to maintain data integrity.

In [ ]:
# checking missing values again .
df.isnull().sum()

,0
Customer_ID,0
Month,0
Age,0
Occupation,0
Annual_Income,0
Monthly_Inhand_Salary,0
Num_Bank_Accounts,0
Num_Credit_Card,0
Interest_Rate,0
Num_of_Loan,0


## Customer-Level Aggregation

The dataset contained 8 monthly records per customer.
To build a decision-support risk assessment system,
a consolidated customer-level dataset was required.

The following aggregation logic was applied:

- Income and financial metrics → Mean
- Delayed payments → Sum
- Loan counts → Max
- Credit history age → Max
- Categorical variables → Mode

This created a single risk profile per customer.

In [ ]:
customer_df = df.groupby('Customer_ID').agg({

    # Financial
    'Annual_Income': 'mean',
    'Monthly_Inhand_Salary': 'mean',
    'Monthly_Balance': 'mean',
    'Amount_invested_monthly': 'mean',

    # Debt & Credit
    'Outstanding_Debt': 'mean',
    'Credit_Utilization_Ratio': 'mean',
    'Total_EMI_per_month': 'mean',
    'Interest_Rate': 'mean',

    # Loan & Accounts
    'Num_of_Loan': 'max',
    'Num_Bank_Accounts': 'max',
    'Num_Credit_Card': 'max',
    'Num_Credit_Inquiries': 'sum',

    # Delay Behaviour
    'Delay_from_due_date': 'mean',
    'Num_of_Delayed_Payment': 'sum',

    # Credit History
    'Credit_History_Age': 'max',

    # Categorical
    'Occupation': 'first',
    'Type_of_Loan': 'first',
    'Credit_Mix': lambda x: x.mode()[0],
    'Payment_Behaviour': lambda x: x.mode()[0],
    'Payment_of_Min_Amount': lambda x: x.mode()[0],
    'Credit_Score': lambda x: x.mode()[0]

}).reset_index()

## Feature Engineering

To enhance risk analysis, the following derived features were created:

1. EMI to Income Ratio  
   → Measures financial burden of loan repayments.

2. Income Group (Low / Middle / High)  
   → Segments customers based on annual income.

3. Utilization Band (Low / Medium / High)  
   → Categorizes credit utilization levels.

4. Delay Risk Category  
   → Groups customers based on average payment delays.

5. Composite Risk Score  
   → A weighted score combining:
     - EMI burden
     - Credit utilization
     - Payment delay behavior

6. Risk Category (Low / Medium / High Risk)  
   → Final customer risk segmentation.

In [ ]:
# EMI to Income Ratio
customer_df['EMI_to_Income_Ratio'] = (
    customer_df['Total_EMI_per_month'] /
    customer_df['Monthly_Inhand_Salary']
)

In [ ]:
# Income Group
customer_df['Income_Group'] = pd.cut(
    customer_df['Annual_Income'],
    bins=[0, 25000, 75000, 200000],
    labels=['Low', 'Middle', 'High']
)

In [ ]:
# Utilization Band
customer_df['Utilization_Band'] = pd.cut(
    customer_df['Credit_Utilization_Ratio'],
    bins=[0, 30, 60, 100],
    labels=['Low', 'Medium', 'High']
)

In [ ]:
#delay risk
customer_df['Delay_Risk'] = pd.cut(
    customer_df['Delay_from_due_date'],
    bins=[float ('-inf'), 5, 15, float ('inf')],
    labels=['Low', 'Medium', 'High']
)

## Risk Scoring Logic

A composite risk score was calculated using key financial indicators:

Risk Score =
(EMI to Income Ratio × Weight) +
(Credit Utilization × Weight) +
(Payment Delay × Weight)

This approach helps quantify customer risk
based on financial stress and behavioral patterns.

The final score was categorized into:

- Low Risk
- Medium Risk
- High Risk

This segmentation enables better loan approval decisions.

In [ ]:
# Risk Score
customer_df['Risk_Score'] = (
    customer_df['EMI_to_Income_Ratio'] * 40 +
    (customer_df['Credit_Utilization_Ratio'] / 100) * 30 +
    (customer_df['Delay_from_due_date'] / 30) * 30
)

In [ ]:
# Final Risk Category
customer_df['Risk_Category'] = pd.cut(
    customer_df['Risk_Score'],
    bins=[0, 30, 60, 100],
    labels=['Low Risk', 'Medium Risk', 'High Risk']
)

In [ ]:
# High Risk Flag
customer_df['High_Risk_Flag'] = np.where(
    (customer_df['Credit_Score'] == 'Poor') |
    (customer_df['EMI_to_Income_Ratio'] > 0.08) |
    (customer_df['Credit_Utilization_Ratio'] > 60),
    'High Risk',
    'Normal'
)

In [ ]:
customer_df['Approval_Status'] = np.where(
    customer_df['Risk_Category'] == 'High Risk',
    'Reject',
    np.where(
        customer_df['Risk_Category'] == 'Medium Risk',
        'Review',
        'Approve'
    )
)

In [ ]:
customer_df.info()
customer_df.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2559 entries, 0 to 2558
Data columns (total 30 columns):
 #   Column                    Non-Null Count  Dtype   
---  ------                    --------------  -----   
 0   Customer_ID               2559 non-null   int64   
 1   Annual_Income             2559 non-null   float64 
 2   Monthly_Inhand_Salary     2559 non-null   float64 
 3   Monthly_Balance           2559 non-null   float64 
 4   Amount_invested_monthly   2559 non-null   float64 
 5   Outstanding_Debt          2559 non-null   float64 
 6   Credit_Utilization_Ratio  2559 non-null   float64 
 7   Total_EMI_per_month       2559 non-null   float64 
 8   Interest_Rate             2559 non-null   float64 
 9   Num_of_Loan               2559 non-null   float64 
 10  Num_Bank_Accounts         2559 non-null   float64 
 11  Num_Credit_Card           2559 non-null   float64 
 12  Num_Credit_Inquiries      2559 non-null   float64 
 13  Delay_from_due_date       2559 non-null   float6

,Customer_ID,Annual_Income,Monthly_Inhand_Salary,Monthly_Balance,Amount_invested_monthly,Outstanding_Debt,Credit_Utilization_Ratio,Total_EMI_per_month,Interest_Rate,Num_of_Loan,...,Payment_of_Min_Amount,Credit_Score,EMI_to_Income_Ratio,Income_Group,Utilization_Band,Delay_Risk,Risk_Score,Risk_Category,High_Risk_Flag,Approval_Status
0,1006,16756.18,1331.348333,314.626667,45.301068,1941.73,31.727384,27.442089,22.0,2.0,...,Yes,Poor,0.020612,Low,Medium,High,58.342705,Medium Risk,High Risk,Review
1,1030,122942.28,10205.190000,716.272375,116.262491,924.10,35.010903,101.567977,3.0,1.0,...,No,Good,0.009953,High,Medium,Medium,18.901374,Low Risk,Normal,Approve
2,1039,72588.99,6242.082500,591.501766,42.321092,1031.21,32.819247,72.928311,19.0,2.0,...,No,Good,0.011683,Middle,Medium,High,32.813107,Medium Risk,Normal,Review
3,1048,14924.12,1347.676667,304.990704,21.904032,22.71,32.430737,19.928365,16.0,2.0,...,Yes,Standard,0.014787,Low,Medium,Medium,17.195709,Low Risk,Normal,Approve
4,1061,60572.70,4844.725000,466.932665,56.199333,1582.55,29.718031,88.716042,20.0,2.0,...,Yes,Standard,0.018312,Middle,Low,High,39.772885,Medium Risk,Normal,Review


## Business Objectives of Dashboard

The Power BI dashboard aims to answer:

- What percentage of customers are high risk?
- How does income impact credit score?
- Does high credit utilization increase default risk?
- What is the impact of delayed payments?
- Which customer segments should be approved or rejected?

The dashboard provides visual insights for data-driven decision making.

Export for Power BI

In [ ]:
customer_df.to_csv("credit_risk_customer_level.csv", index=False)

## Conclusion

This project demonstrates how raw transactional credit data
can be transformed into a structured customer-level
risk assessment system.

By combining data cleaning, feature engineering, and risk scoring,
a practical decision-support framework was developed.

The final Power BI dashboard enables financial institutions
to reduce risk exposure and optimize loan approvals.